# Triton Grammar — XGrammar Constrained Decoding

Este notebook demuestra cómo usar `triton_kernel.ebnf` con XGrammar para forzar que
un LLM genere **únicamente código Python/Triton sintácticamente válido**.

Secciones:
1. Setup e instalación
2. Cargar y validar la gramática
3. Integración HuggingFace (demo local, CPU/GPU)
4. Loop manual del GrammarMatcher
5. Snippet para vLLM (GPU production)

## 1. Setup

In [ ]:
%pip install --quiet "xgrammar>=0.1.30" "transformers>=4.45" "torch>=2.2"

In [ ]:
import xgrammar as xgr
import torch
from pathlib import Path

print("torch:", torch.__version__)
print("CUDA: ", torch.cuda.is_available())

## 2. Cargar y validar la gramática

La gramática vive en `grammars/triton_kernel.ebnf`.  
Aquí la cargamos como string y la compilamos contra el vocabulario del modelo.

In [ ]:
# Ajusta la ruta si corres desde otro directorio
GRAMMAR_PATH = Path("../grammars/triton_kernel.ebnf")
GRAMMAR_STR  = GRAMMAR_PATH.read_text(encoding="utf-8")

print(f"Gramática cargada: {len(GRAMMAR_STR)} caracteres")
print("\n--- Primeras 20 líneas ---")
print("\n".join(GRAMMAR_STR.splitlines()[:20]))

### 2a. ¿Por qué esta gramática es diferente de triton.y?

| Aspecto | triton.y (YACC/Bison) | triton_kernel.ebnf (XGrammar) |
|---|---|---|
| Dirección | Bottom-up (LALR) | Top-down (LL / PDA) |
| Terminales | Códigos de token (`KW_IF`, `IDENTIFIER`) | Patrones de texto (`"if "`, `[a-zA-Z_]`) |
| Indentación | INDENT/DEDENT (tokens del lexer) | Espacios literales (`"    "`) |
| Recursión izquierda | Permitida (LR la maneja) | **Eliminada** (LL loop infinito) |
| Propósito | Verificar código existente | Guiar generación carácter a carácter |

## 3. Integración HuggingFace — LogitsProcessor

El `LogitsProcessor` aplica la máscara de XGrammar **en cada paso** de `model.generate()`.  
Para la demo usamos un modelo pequeño que corre en CPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Cambia a un modelo de código más grande si tienes GPU:
# MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL = "codellama/CodeLlama-7b-Instruct-hf"
MODEL  = "Salesforce/codegen-350M-mono"   # pequeño, corre en CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

model     = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, device_map=device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config    = AutoConfig.from_pretrained(MODEL)

print(f"Cargado {MODEL} en {device}. Vocab size: {config.vocab_size}")

In [ ]:
# Compilar la gramática Triton contra el vocabulario del modelo
# Este paso tarda ~5-30 s la primera vez; el resultado es cacheable.

tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
compiler       = xgr.GrammarCompiler(tokenizer_info)
compiled       = compiler.compile_grammar(GRAMMAR_STR)

print("Gramática compilada:", compiled)

In [ ]:
# Prompt: pedirle al modelo un kernel de suma vectorial
prompt = (
    "# Triton kernel: element-wise addition of two 1-D float32 tensors\n"
    "import triton\n"
    "import triton.language as tl\n"
    "\n"
    "@triton.jit\n"
    "def "
)

inputs          = tokenizer(prompt, return_tensors="pt").to(device)
logits_proc     = xgr.contrib.hf.LogitsProcessor(compiled)

out_ids = model.generate(
    **inputs,
    max_new_tokens=300,
    logits_processors=[logits_proc],
    do_sample=False,
)

generated = tokenizer.decode(out_ids[0], skip_special_tokens=True)
print("=" * 60)
print(generated)
print("=" * 60)

In [ ]:
# Verificación: ¿el output es Python válido?
try:
    compile(generated, "<generated>", "exec")
    print("✓ Python syntax: OK")
except SyntaxError as e:
    print(f"✗ SyntaxError: {e}")

## 4. Loop manual del GrammarMatcher

Mismo algoritmo que `LogitsProcessor` pero explícito.  
Útil para entender qué pasa en cada paso:

```
fill_next_token_bitmask  →  consulta la cache adaptativa
apply_token_bitmask      →  pone -inf en tokens inválidos
accept_token             →  avanza el PDA al siguiente estado
```

In [ ]:
matcher      = xgr.GrammarMatcher(compiled)
token_bitmask = xgr.allocate_token_bitmask(1, tokenizer_info.vocab_size)

print(f"Bitmask shape: {tuple(token_bitmask.shape)}, dtype: {token_bitmask.dtype}")
print(f"Bytes/request/step: {token_bitmask.element_size() * token_bitmask.numel()} B")

In [ ]:
prompt      = "import triton\nimport triton.language as tl\n\n@triton.jit\ndef "
inputs      = tokenizer(prompt, return_tensors="pt").to(device)
input_ids   = inputs.input_ids[0].tolist()
prompt_len  = len(input_ids)

matcher.reset()
step, max_steps = 0, 400

while not matcher.is_terminated() and step < max_steps:
    with torch.no_grad():
        logits = model(torch.tensor([input_ids], device=device)).logits

    # 1. Obtener máscara de tokens válidos dado el estado actual del PDA
    matcher.fill_next_token_bitmask(token_bitmask)

    # 2. Aplicar máscara: tokens inválidos → -inf
    xgr.apply_token_bitmask_inplace(logits[0, -1, :], token_bitmask.to(device))

    # 3. Elegir el token de mayor probabilidad (greedy)
    next_id = torch.argmax(logits[0, -1, :]).item()

    # 4. Avanzar el PDA al siguiente estado
    matcher.accept_token(next_id)

    input_ids.append(next_id)
    step += 1

print(f"Generados {step} tokens.")
print(tokenizer.decode(input_ids[prompt_len:], skip_special_tokens=True))

## 5. vLLM — production (GPU)

Cuando el modelo corre en un servidor vLLM, se pasa la gramática via `extra_body`.
Este es el path para integrar con el pipeline de `GrammarConstraint-KernelGeneration`.

In [ ]:
# Levantar el servidor (en otra terminal / contenedor Modal):
# 
#   vllm serve mistralai/devstral-small-2507 \
#        --guided-decoding-backend xgrammar \
#        --port 8000

# Luego usar el provider del proyecto:

# from openai import OpenAI
# grammar_str = open("../grammars/triton_kernel.ebnf").read()
#
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="-")
# response = client.chat.completions.create(
#     model="mistralai/devstral-small-2507",
#     messages=[{"role": "user", "content": "Write a Triton kernel for vector addition."}],
#     extra_body={
#         "guided_grammar": grammar_str,          # <-- nuestra gramática
#         "guided_decoding_backend": "xgrammar",  # fuerza XGrammar
#     },
# )
# print(response.choices[0].message.content)

print("Descomenta y corre contra un servidor vLLM con GPU.")

## Apéndice — Mapa gramática YACC ↔ XGrammar

```
triton.y (YACC)                    triton_kernel.ebnf (XGrammar)
─────────────────────────────────────────────────────────────────
KW_IF                          →   "if "
IDENTIFIER                     →   [a-zA-Z_][a-zA-Z0-9_]*
INT_LITERAL                    →   [1-9][0-9]* | "0"
NEWLINE · INDENT · DEDENT      →   "\n" + "    " (literal)
if_stmt → KW_IF expr COLON     →   "if " expr ":\n" suite2
         suite elif_chain               body2_stmt+
primary (left-recursive)       →   primary_expr ::= atom suffix*
expr → expr OP_PLUS expr       →   add_expr ::= mul_expr
  (precedencia via %left)              (" + " | " - ") mul_expr)*
```